In [ ]:
from pathlib import Path
from contextlib import nullcontext
from dataclasses import dataclass
from types import SimpleNamespace
import json
import math
import random
import time
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from torch.nn.functional import scaled_dot_product_attention

from transformers import AutoModel

ARCHITECTURE_CHOICES = (
    "bert_encoder_trm",
    "bert_embeddings_trm",
    "bert_encoder_rerank_head",
    "vanilla_transformer",
)

ARCHITECTURE = "bert_encoder_trm"

SEED = 13
PRECISION = "bf16-mixed"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

SEQ_LEN = 256
MAX_QUERY_LEN = 32
MAX_DOC_LEN = 221
VOCAB_SIZE = 30_522
CLS_ID = 101
SEP_ID = 102
PAD_ID = 0

LATENCY_BATCH_SIZE = 1
LATENCY_WARMUP_STEPS = 10
LATENCY_MEASURE_STEPS = 50

BERT_MODEL_NAME = "bert-base-uncased"
# Path to the local bert-base-uncased checkpoint directory.
BERT_LOCAL_PATH = Path("..") / "bert_base_artifacts"
# Directory for generated latency JSON files.
OUTPUT_DIR = Path.cwd()

HIDDEN_SIZE = 512
NUM_HEADS = 8
L_LAYERS = 2
H_CYCLES = 2
L_CYCLES = 4
HALT_MAX_STEPS = 1
HALT_EXPLORATION_PROB = 0.0
EXPANSION = 4.0
VANILLA_TRANSFORMER_LAYERS = 2
SCORING_HEAD_DROPOUT = 0.1
FREEZE_ENCODER = True
ADD_TRM_SEGMENT_EMBEDDINGS = True
FREEZE_PRETRAINED_WORD_EMBEDDINGS = True
PRETRAINED_WORD_EMBEDDING_PROJECTION = True

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def resolve_precision(requested_precision: str, device_kind: str) -> str:
    if requested_precision == "bf16-mixed":
        if device_kind == "cuda" and torch.cuda.is_available() and torch.cuda.is_bf16_supported():
            return requested_precision
        print(f'Falling back from precision={requested_precision!r} to "32-true" because bf16 is not available.')
        return "32-true"
    if requested_precision == "16-mixed" and device_kind != "cuda":
        print(f'Falling back from precision={requested_precision!r} to "32-true" because fp16 autocast needs CUDA.')
        return "32-true"
    return requested_precision

def setup_device(requested_device: str) -> torch.device:
    if requested_device.startswith("cuda") and not torch.cuda.is_available():
        return torch.device("cpu")
    return torch.device(requested_device)

def get_autocast_context(device: torch.device, precision: str):
    if device.type != "cuda":
        return nullcontext()
    if precision == "bf16-mixed":
        return torch.autocast(device_type="cuda", dtype=torch.bfloat16)
    if precision == "16-mixed":
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return nullcontext()

def model_device(model: nn.Module) -> torch.device:
    return next(model.parameters()).device

def model_autocast_context(model: nn.Module):
    return get_autocast_context(model_device(model), EFFECTIVE_PRECISION)

def count_model_parameters(model: nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters())

def count_trainable_parameters(model: nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

seed_everything(SEED)
device = setup_device(DEVICE)
EFFECTIVE_PRECISION = resolve_precision(PRECISION, device.type)
MODEL_FORWARD_DTYPE = "bfloat16" if EFFECTIVE_PRECISION == "bf16-mixed" else "float32"

if ARCHITECTURE not in ARCHITECTURE_CHOICES:
    raise ValueError(f"Unknown ARCHITECTURE={ARCHITECTURE!r}; choose one of {ARCHITECTURE_CHOICES}.")

print(json.dumps({
    "architecture": ARCHITECTURE,
    "device": str(device),
    "precision": EFFECTIVE_PRECISION,
    "seq_len": SEQ_LEN,
    "max_query_len": MAX_QUERY_LEN,
    "max_doc_len": MAX_DOC_LEN,
    "vocab_size": VOCAB_SIZE,
    "latency_batch_size": LATENCY_BATCH_SIZE,
    "latency_warmup_steps": LATENCY_WARMUP_STEPS,
    "latency_measure_steps": LATENCY_MEASURE_STEPS,
    "bert_local_path": str(BERT_LOCAL_PATH),
}, indent=2))


In [ ]:
CosSin = Tuple[torch.Tensor, torch.Tensor]

def trunc_normal_init_(tensor: torch.Tensor, std: float = 1.0, lower: float = -2.0, upper: float = 2.0):
    with torch.no_grad():
        if std == 0:
            tensor.zero_()
        else:
            sqrt2 = math.sqrt(2)
            a = math.erf(lower / sqrt2)
            b = math.erf(upper / sqrt2)
            z = (b - a) / 2

            c = (2 * math.pi) ** -0.5
            pdf_u = c * math.exp(-0.5 * lower ** 2)
            pdf_l = c * math.exp(-0.5 * upper ** 2)
            comp_std = std / math.sqrt(1 - (upper * pdf_u - lower * pdf_l) / z - ((pdf_u - pdf_l) / z) ** 2)

            tensor.uniform_(a, b)
            tensor.erfinv_()
            tensor.mul_(sqrt2 * comp_std)
            tensor.clip_(lower * comp_std, upper * comp_std)
    return tensor

def _find_multiple(a: int, b: int) -> int:
    return (-(a // -b)) * b

def rotate_half(x: torch.Tensor) -> torch.Tensor:
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_pos_emb(q: torch.Tensor, k: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor):
    seq_len = q.shape[1]
    cos = cos[:seq_len]
    sin = sin[:seq_len]
    orig_dtype = q.dtype
    q = q.to(cos.dtype)
    k = k.to(cos.dtype)
    q_embed = (q * cos.unsqueeze(-2)) + (rotate_half(q) * sin.unsqueeze(-2))
    k_embed = (k * cos.unsqueeze(-2)) + (rotate_half(k) * sin.unsqueeze(-2))
    return q_embed.to(orig_dtype), k_embed.to(orig_dtype)

class CastedLinear(nn.Module):
    def __init__(self, in_features: int, out_features: int, bias: bool):
        super().__init__()
        self.weight = nn.Parameter(trunc_normal_init_(torch.empty((out_features, in_features)), std=1.0 / (in_features ** 0.5)))
        self.bias = nn.Parameter(torch.zeros((out_features,))) if bias else None

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        bias = self.bias.to(inputs.dtype) if self.bias is not None else None
        return F.linear(inputs, self.weight.to(inputs.dtype), bias=bias)

class CastedEmbedding(nn.Module):
    def __init__(self, num_embeddings: int, embedding_dim: int, init_std: float, cast_to: torch.dtype):
        super().__init__()
        self.cast_to = cast_to
        self.embedding_weight = nn.Parameter(trunc_normal_init_(torch.empty((num_embeddings, embedding_dim)), std=init_std))

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        return F.embedding(inputs, self.embedding_weight.to(self.cast_to))

class RotaryEmbedding(nn.Module):
    def __init__(self, dim: int, max_position_embeddings: int, base: float):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim))
        t = torch.arange(max_position_embeddings, dtype=torch.float32)
        freqs = torch.outer(t, inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer('cos_cached', emb.cos(), persistent=False)
        self.register_buffer('sin_cached', emb.sin(), persistent=False)

    def forward(self) -> CosSin:
        return self.cos_cached, self.sin_cached

class Attention(nn.Module):
    def __init__(self, hidden_size: int, head_dim: int, num_heads: int, num_key_value_heads: int, causal: bool = False):
        super().__init__()
        self.hidden_size = hidden_size
        self.head_dim = head_dim
        self.output_size = head_dim * num_heads
        self.num_heads = num_heads
        self.num_key_value_heads = num_key_value_heads
        self.causal = causal
        self.qkv_proj = CastedLinear(hidden_size, (num_heads + 2 * num_key_value_heads) * head_dim, bias=False)
        self.o_proj = CastedLinear(self.output_size, hidden_size, bias=False)

    def forward(self, cos_sin: CosSin, hidden_states: torch.Tensor, attention_mask: torch.Tensor = None) -> torch.Tensor:
        batch_size, seq_len, _ = hidden_states.shape
        qkv = self.qkv_proj(hidden_states)
        qkv = qkv.view(batch_size, seq_len, self.num_heads + 2 * self.num_key_value_heads, self.head_dim)
        query = qkv[:, :, : self.num_heads]
        key = qkv[:, :, self.num_heads : self.num_heads + self.num_key_value_heads]
        value = qkv[:, :, self.num_heads + self.num_key_value_heads :]
        if cos_sin is not None:
            cos, sin = cos_sin
            query, key = apply_rotary_pos_emb(query, key, cos, sin)
        query = query.permute(0, 2, 1, 3)
        key = key.permute(0, 2, 1, 3)
        value = value.permute(0, 2, 1, 3)
        attn_mask = attention_mask.to(query.dtype) if attention_mask is not None else None
        attn_output = scaled_dot_product_attention(query=query, key=key, value=value, attn_mask=attn_mask, is_causal=self.causal)
        attn_output = attn_output.permute(0, 2, 1, 3).reshape(batch_size, seq_len, self.output_size)
        return self.o_proj(attn_output)

class SwiGLU(nn.Module):
    def __init__(self, hidden_size: int, expansion: float):
        super().__init__()
        inter = _find_multiple(round(expansion * hidden_size * 2 / 3), 256)
        self.gate_up_proj = CastedLinear(hidden_size, inter * 2, bias=False)
        self.down_proj = CastedLinear(inter, hidden_size, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        gate, up = self.gate_up_proj(x).chunk(2, dim=-1)
        return self.down_proj(F.silu(gate) * up)

def rms_norm(hidden_states: torch.Tensor, variance_epsilon: float) -> torch.Tensor:
    input_dtype = hidden_states.dtype
    hidden_states = hidden_states.to(torch.float32)
    variance = hidden_states.square().mean(-1, keepdim=True)
    hidden_states = hidden_states * torch.rsqrt(variance + variance_epsilon)
    return hidden_states.to(input_dtype)


In [ ]:
@dataclass
class TrmInnerCarry:
    z_H: torch.Tensor
    z_L: torch.Tensor

@dataclass
class TrmCarry:
    inner_carry: TrmInnerCarry
    steps: torch.Tensor
    halted: torch.Tensor
    current_data: Dict[str, torch.Tensor]

@dataclass
class TrmRerankerConfig:
    batch_size: int
    seq_len: int
    vocab_size: int
    H_cycles: int
    L_cycles: int
    H_layers: int
    L_layers: int
    hidden_size: int
    expansion: float
    num_heads: int
    pos_encodings: str
    rms_norm_eps: float = 1e-5
    rope_theta: float = 10000.0
    halt_max_steps: int = 1
    halt_exploration_prob: float = 0.0
    forward_dtype: str = 'float32'
    mlp_t: bool = False
    no_ACT_continue: bool = True
    num_segment_types: int = 3
    input_embedding_backend: str = "scratch"
    pretrained_embedding_model_name: str = "bert-base-uncased"
    pretrained_embedding_dim: Optional[int] = None
    freeze_pretrained_word_embeddings: bool = True
    pretrained_word_embedding_projection: bool = True
    encoder_name: Optional[str] = None
    encoder_local_path: Optional[str] = None
    freeze_encoder: bool = True
    add_trm_segment_embeddings: bool = True

class FrozenPretrainedProjectedWordEmbeddings(nn.Module):
    def __init__(self, config: TrmRerankerConfig):
        super().__init__()
        self.config = config
        self.forward_dtype = getattr(torch, config.forward_dtype)
        if not config.freeze_pretrained_word_embeddings:
            raise ValueError("Variant C requires frozen pretrained word embeddings; set freeze_pretrained_word_embeddings=True.")
        if not config.pretrained_word_embedding_projection:
            raise ValueError("Variant C requires the pretrained word embedding projection to be enabled.")

        pretrained_source = str(Path(config.pretrained_embedding_model_name).expanduser())
        pretrained_model = AutoModel.from_pretrained(pretrained_source, local_files_only=True)
        weight = pretrained_model.embeddings.word_embeddings.weight.detach().clone()
        if weight.shape[0] != config.vocab_size:
            raise ValueError(
                f"Pretrained embedding vocab size {weight.shape[0]} does not match tokenizer vocab size {config.vocab_size}. "
                f"pretrained_embedding_model_name={config.pretrained_embedding_model_name!r}"
            )

        self.pretrained_embedding_model_name = config.pretrained_embedding_model_name
        self.pretrained_embedding_source = pretrained_source
        self.pretrained_embedding_dim = int(weight.shape[1])
        self.config.pretrained_embedding_dim = self.pretrained_embedding_dim
        self.register_buffer("embedding_weight", weight, persistent=False)
        self.proj = CastedLinear(self.pretrained_embedding_dim, config.hidden_size, bias=False)
        del pretrained_model

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        x = F.embedding(input_ids.to(torch.int64), self.embedding_weight)
        x = x.to(self.forward_dtype)
        return self.proj(x)

class TrmBlock(nn.Module):
    def __init__(self, config: TrmRerankerConfig) -> None:
        super().__init__()
        self.config = config
        if self.config.mlp_t:
            self.mlp_t = SwiGLU(hidden_size=self.config.seq_len, expansion=config.expansion)
        else:
            self.self_attn = Attention(
                hidden_size=config.hidden_size,
                head_dim=config.hidden_size // config.num_heads,
                num_heads=config.num_heads,
                num_key_value_heads=config.num_heads,
                causal=False,
            )
        self.mlp = SwiGLU(hidden_size=config.hidden_size, expansion=config.expansion)
        self.norm_eps = config.rms_norm_eps

    def forward(self, cos_sin: CosSin, hidden_states: torch.Tensor, attention_mask: torch.Tensor = None) -> torch.Tensor:
        if self.config.mlp_t:
            hidden_states = hidden_states.transpose(1, 2)
            out = self.mlp_t(hidden_states)
            hidden_states = rms_norm(hidden_states + out, variance_epsilon=self.norm_eps)
            hidden_states = hidden_states.transpose(1, 2)
        else:
            hidden_states = rms_norm(
                hidden_states + self.self_attn(cos_sin=cos_sin, hidden_states=hidden_states, attention_mask=attention_mask),
                variance_epsilon=self.norm_eps,
            )
        out = self.mlp(hidden_states)
        hidden_states = rms_norm(hidden_states + out, variance_epsilon=self.norm_eps)
        return hidden_states

class TrmReasoningModule(nn.Module):
    def __init__(self, layers: List[TrmBlock]):
        super().__init__()
        self.layers = nn.ModuleList(layers)

    def forward(self, hidden_states: torch.Tensor, input_injection: torch.Tensor, attention_mask: torch.Tensor = None, **kwargs) -> torch.Tensor:
        hidden_states = hidden_states + input_injection
        for layer in self.layers:
            hidden_states = layer(hidden_states=hidden_states, attention_mask=attention_mask, **kwargs)
        return hidden_states

class TrmRerankerInner(nn.Module):
    def __init__(self, config: TrmRerankerConfig) -> None:
        super().__init__()
        self.config = config
        self.forward_dtype = getattr(torch, self.config.forward_dtype)
        self.embed_scale = math.sqrt(self.config.hidden_size)
        embed_init_std = 1.0 / self.embed_scale

        self.encoder = None
        self.use_pretrained_encoder = self.config.encoder_name is not None

        if self.use_pretrained_encoder:
            encoder_name_or_path = self.config.encoder_local_path or self.config.encoder_name
            encoder_kwargs = {"local_files_only": True} if self.config.encoder_local_path else {}
            self.encoder = AutoModel.from_pretrained(encoder_name_or_path, **encoder_kwargs)
            if self.config.encoder_name is not None:
                self.encoder.config._name_or_path = self.config.encoder_name
            if self.config.freeze_encoder:
                for parameter in self.encoder.parameters():
                    parameter.requires_grad_(False)
            encoder_hidden_size = int(self.encoder.config.hidden_size)
            if encoder_hidden_size != self.config.hidden_size:
                self.encoder_proj = CastedLinear(encoder_hidden_size, self.config.hidden_size, bias=False)
            else:
                self.encoder_proj = nn.Identity()
            self.encoder_norm = nn.LayerNorm(self.config.hidden_size)
        elif self.config.input_embedding_backend == "scratch":
            self.embed_tokens = CastedEmbedding(
                self.config.vocab_size,
                self.config.hidden_size,
                init_std=embed_init_std,
                cast_to=self.forward_dtype,
            )
        elif self.config.input_embedding_backend == "bert_word_embeddings_projected":
            self.embed_tokens = FrozenPretrainedProjectedWordEmbeddings(self.config)
        else:
            raise ValueError(f"Unknown input_embedding_backend={self.config.input_embedding_backend!r}")

        self.segment_emb = CastedEmbedding(self.config.num_segment_types, self.config.hidden_size, init_std=embed_init_std, cast_to=self.forward_dtype)
        self.score_head = CastedLinear(self.config.hidden_size, 1, bias=True)
        self.q_head = CastedLinear(self.config.hidden_size, 2, bias=True)

        if self.config.pos_encodings == 'rope':
            self.rotary_emb = RotaryEmbedding(
                dim=self.config.hidden_size // self.config.num_heads,
                max_position_embeddings=self.config.seq_len,
                base=self.config.rope_theta,
            )
        elif self.config.pos_encodings == 'learned':
            self.embed_pos = CastedEmbedding(self.config.seq_len, self.config.hidden_size, init_std=embed_init_std, cast_to=self.forward_dtype)

        self.L_level = TrmReasoningModule(
            layers=[TrmBlock(self.config) for _ in range(self.config.L_layers)]
        )

        self.register_buffer('H_init', trunc_normal_init_(torch.empty(self.config.hidden_size, dtype=self.forward_dtype), std=1), persistent=True)
        self.register_buffer('L_init', trunc_normal_init_(torch.empty(self.config.hidden_size, dtype=self.forward_dtype), std=1), persistent=True)

        with torch.no_grad():
            self.q_head.weight.zero_()
            self.q_head.bias.fill_(-5)

    def _input_embeddings(
        self,
        input_ids: torch.Tensor,
        token_type_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        bert_token_type_ids: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        if self.use_pretrained_encoder:
            if bert_token_type_ids is None:
                bert_token_type_ids = (token_type_ids == 2).to(torch.long)
            encoder_kwargs = {
                "input_ids": input_ids.to(torch.long),
                "attention_mask": attention_mask.to(torch.long),
                "token_type_ids": bert_token_type_ids.to(torch.long),
            }
            if self.config.freeze_encoder:
                self.encoder.eval()
                with torch.no_grad():
                    encoder_states = self.encoder(**encoder_kwargs).last_hidden_state
            else:
                encoder_states = self.encoder(**encoder_kwargs).last_hidden_state
            embedding = self.encoder_proj(encoder_states.to(self.forward_dtype))
            if self.config.add_trm_segment_embeddings:
                embedding = embedding + self.segment_emb(token_type_ids.to(torch.long))
            embedding = self.encoder_norm(embedding.float()).to(self.forward_dtype)
            return embedding

        embedding = self.embed_tokens(input_ids.to(torch.int64))
        embedding = embedding + self.segment_emb(token_type_ids.to(torch.int64))
        if self.config.pos_encodings == 'learned':
            embedding = 0.707106781 * (embedding + self.embed_pos.embedding_weight[: input_ids.shape[1]].to(self.forward_dtype))
        return self.embed_scale * embedding

    def _attention_mask(self, attention_mask: torch.Tensor) -> torch.Tensor:
        attention_mask = attention_mask.to(torch.bool)
        additive_mask = torch.zeros(attention_mask.shape, dtype=self.forward_dtype, device=attention_mask.device)
        additive_mask = additive_mask.masked_fill(~attention_mask, torch.finfo(self.forward_dtype).min)
        return additive_mask[:, None, None, :]

    def empty_carry(self, batch_size: int) -> TrmInnerCarry:
        device = self.H_init.device
        return TrmInnerCarry(
            z_H=torch.empty(batch_size, self.config.seq_len, self.config.hidden_size, dtype=self.forward_dtype, device=device),
            z_L=torch.empty(batch_size, self.config.seq_len, self.config.hidden_size, dtype=self.forward_dtype, device=device),
        )

    def reset_carry(self, reset_flag: torch.Tensor, carry: TrmInnerCarry) -> TrmInnerCarry:
        reset_view = reset_flag.view(-1, 1, 1)
        return TrmInnerCarry(
            z_H=torch.where(reset_view, self.H_init, carry.z_H),
            z_L=torch.where(reset_view, self.L_init, carry.z_L),
        )

    def forward(self, carry: TrmInnerCarry, batch: Dict[str, torch.Tensor]):
        seq_info = {
            'cos_sin': self.rotary_emb() if hasattr(self, 'rotary_emb') else None,
            'attention_mask': self._attention_mask(batch['attention_mask']),
        }
        input_embeddings = self._input_embeddings(
            batch['input_ids'],
            batch['token_type_ids'],
            batch['attention_mask'],
            batch.get('bert_token_type_ids'),
        )
        z_H, z_L = carry.z_H, carry.z_L
        with torch.no_grad():
            for _ in range(self.config.H_cycles - 1):
                for _ in range(self.config.L_cycles):
                    z_L = self.L_level(z_L, z_H + input_embeddings, **seq_info)
                z_H = self.L_level(z_H, z_L, **seq_info)
        for _ in range(self.config.L_cycles):
            z_L = self.L_level(z_L, z_H + input_embeddings, **seq_info)
        z_H = self.L_level(z_H, z_L, **seq_info)

        new_carry = TrmInnerCarry(z_H=z_H.detach(), z_L=z_L.detach())
        cls_state = z_H[:, 0]
        scores = self.score_head(cls_state).squeeze(-1)
        q_logits = self.q_head(cls_state).to(torch.float32)
        return new_carry, scores, (q_logits[..., 0], q_logits[..., 1])

class TrmReranker(nn.Module):
    def __init__(self, config_dict: dict):
        super().__init__()
        self.config = TrmRerankerConfig(**config_dict)
        self.inner = TrmRerankerInner(self.config)

    def initial_carry(self, batch: Dict[str, torch.Tensor]) -> TrmCarry:
        batch_size = batch['input_ids'].shape[0]
        return TrmCarry(
            inner_carry=self.inner.empty_carry(batch_size),
            steps=torch.zeros((batch_size,), dtype=torch.int32, device=batch['input_ids'].device),
            halted=torch.ones((batch_size,), dtype=torch.bool, device=batch['input_ids'].device),
            current_data={key: torch.empty_like(value) for key, value in batch.items()},
        )

    def forward(self, carry: TrmCarry, batch: Dict[str, torch.Tensor]):
        new_inner_carry = self.inner.reset_carry(carry.halted, carry.inner_carry)
        new_steps = torch.where(carry.halted, torch.zeros_like(carry.steps), carry.steps)
        new_current_data = {
            key: torch.where(carry.halted.view((-1,) + (1,) * (value.ndim - 1)), batch[key], value)
            for key, value in carry.current_data.items()
        }
        new_inner_carry, scores, (q_halt_logits, q_continue_logits) = self.inner(new_inner_carry, new_current_data)
        outputs = {
            'scores': scores,
            'q_halt_logits': q_halt_logits,
            'q_continue_logits': q_continue_logits,
        }
        with torch.no_grad():
            new_steps = new_steps + 1
            is_last_step = new_steps >= self.config.halt_max_steps
            halted = is_last_step
            if self.training and (self.config.halt_max_steps > 1):
                if self.config.no_ACT_continue:
                    halted = halted | (q_halt_logits > 0)
                else:
                    halted = halted | (q_halt_logits > q_continue_logits)
                min_halt_steps = (torch.rand_like(q_halt_logits) < self.config.halt_exploration_prob) * torch.randint_like(
                    new_steps, low=2, high=self.config.halt_max_steps + 1
                )
                halted = halted & (new_steps >= min_halt_steps)
                if not self.config.no_ACT_continue:
                    _, _, (next_q_halt_logits, next_q_continue_logits) = self.inner(new_inner_carry, new_current_data)
                    outputs['target_q_continue'] = torch.sigmoid(
                        torch.where(is_last_step, next_q_halt_logits, torch.maximum(next_q_halt_logits, next_q_continue_logits))
                    )
        return TrmCarry(new_inner_carry, new_steps, halted, new_current_data), outputs

class BertEncoderOnlyReranker(nn.Module):
    def __init__(
        self,
        encoder_name: str,
        encoder_local_path: Optional[str] = None,
        freeze_encoder: bool = True,
        dropout: float = 0.1,
    ):
        super().__init__()

        encoder_name_or_path = encoder_local_path or encoder_name
        encoder_kwargs = {"local_files_only": True} if encoder_local_path else {}

        self.encoder = AutoModel.from_pretrained(encoder_name_or_path, **encoder_kwargs)
        self.encoder.config._name_or_path = encoder_name
        self.freeze_encoder = bool(freeze_encoder)

        if self.freeze_encoder:
            for parameter in self.encoder.parameters():
                parameter.requires_grad_(False)

        hidden_size = int(self.encoder.config.hidden_size)
        self.norm = nn.LayerNorm(hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.score_head = nn.Linear(hidden_size, 1)

        self.config = SimpleNamespace(
            halt_max_steps=1,
            architecture="bert_encoder_only_reranker",
            encoder_name=encoder_name,
            encoder_local_path=encoder_local_path,
            freeze_encoder=self.freeze_encoder,
            hidden_size=hidden_size,
            dropout=float(dropout),
        )

    def initial_carry(self, batch: Dict[str, torch.Tensor]):
        return None

    def _bert_token_type_ids(self, batch: Dict[str, torch.Tensor]) -> torch.Tensor:
        if "bert_token_type_ids" in batch:
            return batch["bert_token_type_ids"].to(torch.long)
        if "token_type_ids" in batch:
            token_type_ids = batch["token_type_ids"].to(torch.long)
            return (token_type_ids == 2).to(torch.long)
        return torch.zeros_like(batch["input_ids"], dtype=torch.long)

    def forward(self, carry, batch: Dict[str, torch.Tensor]):
        encoder_kwargs = {
            "input_ids": batch["input_ids"].to(torch.long),
            "attention_mask": batch["attention_mask"].to(torch.long),
            "token_type_ids": self._bert_token_type_ids(batch),
        }

        if self.freeze_encoder:
            self.encoder.eval()
            with torch.no_grad():
                encoder_outputs = self.encoder(**encoder_kwargs)
        else:
            encoder_outputs = self.encoder(**encoder_kwargs)

        cls_state = encoder_outputs.last_hidden_state[:, 0]
        cls_state = self.norm(cls_state)
        cls_state = self.dropout(cls_state)
        scores = self.score_head(cls_state).squeeze(-1)

        return None, {"scores": scores}

@dataclass
class VanillaTransformerRerankerCarry:
    halted: torch.Tensor

@dataclass
class VanillaTransformerRerankerConfig:
    batch_size: int
    seq_len: int
    vocab_size: int
    num_layers: int
    hidden_size: int
    expansion: float
    num_heads: int
    pos_encodings: str
    rms_norm_eps: float = 1e-5
    rope_theta: float = 10000.0
    halt_max_steps: int = 1
    forward_dtype: str = 'float32'
    num_segment_types: int = 3

class VanillaTransformerBlock(nn.Module):
    def __init__(self, config: VanillaTransformerRerankerConfig) -> None:
        super().__init__()
        self.self_attn = Attention(
            hidden_size=config.hidden_size,
            head_dim=config.hidden_size // config.num_heads,
            num_heads=config.num_heads,
            num_key_value_heads=config.num_heads,
            causal=False,
        )
        self.mlp = SwiGLU(hidden_size=config.hidden_size, expansion=config.expansion)
        self.norm_eps = config.rms_norm_eps

    def forward(self, hidden_states: torch.Tensor, cos_sin: CosSin = None, attention_mask: torch.Tensor = None) -> torch.Tensor:
        hidden_states = rms_norm(
            hidden_states + self.self_attn(cos_sin=cos_sin, hidden_states=hidden_states, attention_mask=attention_mask),
            variance_epsilon=self.norm_eps,
        )
        hidden_states = rms_norm(hidden_states + self.mlp(hidden_states), variance_epsilon=self.norm_eps)
        return hidden_states

class VanillaTransformerRerankerInner(nn.Module):
    def __init__(self, config: VanillaTransformerRerankerConfig) -> None:
        super().__init__()
        self.config = config
        self.forward_dtype = getattr(torch, self.config.forward_dtype)
        self.embed_scale = math.sqrt(self.config.hidden_size)
        embed_init_std = 1.0 / self.embed_scale

        self.embed_tokens = CastedEmbedding(self.config.vocab_size, self.config.hidden_size, init_std=embed_init_std, cast_to=self.forward_dtype)
        self.segment_emb = CastedEmbedding(self.config.num_segment_types, self.config.hidden_size, init_std=embed_init_std, cast_to=self.forward_dtype)
        self.score_head = CastedLinear(self.config.hidden_size, 1, bias=True)
        self.layers = nn.ModuleList([VanillaTransformerBlock(self.config) for _ in range(self.config.num_layers)])

        if self.config.pos_encodings == 'rope':
            self.rotary_emb = RotaryEmbedding(
                dim=self.config.hidden_size // self.config.num_heads,
                max_position_embeddings=self.config.seq_len,
                base=self.config.rope_theta,
            )
        elif self.config.pos_encodings == 'learned':
            self.embed_pos = CastedEmbedding(self.config.seq_len, self.config.hidden_size, init_std=embed_init_std, cast_to=self.forward_dtype)

    def _input_embeddings(self, input_ids: torch.Tensor, token_type_ids: torch.Tensor) -> torch.Tensor:
        embedding = self.embed_tokens(input_ids.to(torch.int64))
        embedding = embedding + self.segment_emb(token_type_ids.to(torch.int64))
        if self.config.pos_encodings == 'learned':
            embedding = 0.707106781 * (embedding + self.embed_pos.embedding_weight[: input_ids.shape[1]].to(self.forward_dtype))
        return self.embed_scale * embedding

    def _attention_mask(self, attention_mask: torch.Tensor) -> torch.Tensor:
        attention_mask = attention_mask.to(torch.bool)
        additive_mask = torch.zeros(attention_mask.shape, dtype=self.forward_dtype, device=attention_mask.device)
        additive_mask = additive_mask.masked_fill(~attention_mask, torch.finfo(self.forward_dtype).min)
        return additive_mask[:, None, None, :]

    def forward(self, batch: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
        cos_sin = self.rotary_emb() if hasattr(self, 'rotary_emb') else None
        attention_mask = self._attention_mask(batch['attention_mask'])
        hidden_states = self._input_embeddings(batch['input_ids'], batch['token_type_ids'])
        for layer in self.layers:
            hidden_states = layer(hidden_states=hidden_states, cos_sin=cos_sin, attention_mask=attention_mask)
        cls_state = hidden_states[:, 0]
        scores = self.score_head(cls_state).squeeze(-1)
        return {'scores': scores}

class VanillaTransformerReranker(nn.Module):
    def __init__(self, config_dict: dict):
        super().__init__()
        self.config = VanillaTransformerRerankerConfig(**config_dict)
        self.inner = VanillaTransformerRerankerInner(self.config)

    def initial_carry(self, batch: Dict[str, torch.Tensor]) -> VanillaTransformerRerankerCarry:
        batch_size = batch['input_ids'].shape[0]
        return VanillaTransformerRerankerCarry(
            halted=torch.ones((batch_size,), dtype=torch.bool, device=batch['input_ids'].device),
        )

    def forward(self, carry: VanillaTransformerRerankerCarry, batch: Dict[str, torch.Tensor]):
        del carry
        outputs = self.inner(batch)
        batch_size = batch['input_ids'].shape[0]
        new_carry = VanillaTransformerRerankerCarry(
            halted=torch.ones((batch_size,), dtype=torch.bool, device=batch['input_ids'].device),
        )
        return new_carry, outputs


In [ ]:
def make_synthetic_batch(batch_size: int, seq_len: int, vocab_size: int, device: torch.device) -> Dict[str, torch.Tensor]:
    input_ids = torch.randint(999, vocab_size, (batch_size, seq_len), dtype=torch.long, device=device)
    input_ids[:, 0] = CLS_ID
    input_ids[:, MAX_QUERY_LEN + 1] = SEP_ID
    input_ids[:, -1] = SEP_ID

    token_type_ids = torch.zeros((batch_size, seq_len), dtype=torch.long, device=device)
    token_type_ids[:, 1:MAX_QUERY_LEN + 1] = 1
    token_type_ids[:, MAX_QUERY_LEN + 2:-1] = 2

    attention_mask = torch.ones((batch_size, seq_len), dtype=torch.long, device=device)
    bert_token_type_ids = (token_type_ids == 2).to(torch.long)
    return {
        "input_ids": input_ids,
        "token_type_ids": token_type_ids,
        "bert_token_type_ids": bert_token_type_ids,
        "attention_mask": attention_mask,
    }

def synchronize_for_latency(device: torch.device) -> bool:
    if device.type == "cuda" and torch.cuda.is_available():
        torch.cuda.synchronize(device)
        return True
    return False

def run_forward_only(model: nn.Module, batch: Dict[str, torch.Tensor], carry=None) -> int:
    if carry is None:
        carry = model.initial_carry(batch)

    outputs = None
    forward_calls = 0
    halt_max_steps = int(getattr(model.config, "halt_max_steps", 1))
    for _ in range(halt_max_steps):
        carry, outputs = model(carry, batch)
        forward_calls += 1
        halted = getattr(carry, "halted", None) if carry is not None else None
        if carry is None or (halted is not None and bool(halted.all())):
            break

    if outputs is None:
        raise RuntimeError("Model produced no outputs")
    if "scores" not in outputs:
        raise RuntimeError(f"Model outputs do not include scores: {outputs.keys()}")
    return forward_calls

def measure_forward_latency(
    model: nn.Module,
    batch: Dict[str, torch.Tensor],
    warmup_steps: int,
    measure_steps: int,
) -> Dict[str, object]:
    warmup_steps = max(0, int(warmup_steps))
    measure_steps = int(measure_steps)
    if measure_steps <= 0:
        raise ValueError("measure_steps must be positive")

    device = model_device(model)
    batch_size = int(batch["input_ids"].shape[0])
    seq_len = int(batch["input_ids"].shape[1])
    halt_max_steps = int(getattr(model.config, "halt_max_steps", 1))
    elapsed_ms: List[float] = []
    forward_calls_per_iteration: List[int] = []
    was_training = model.training

    model.eval()
    try:
        with torch.inference_mode(), model_autocast_context(model):
            for _ in range(warmup_steps):
                run_forward_only(model, batch)

            synchronize_for_latency(device)
            for _ in range(measure_steps):
                carry = model.initial_carry(batch)
                synchronize_for_latency(device)
                start = time.perf_counter()
                forward_calls = run_forward_only(model, batch, carry=carry)
                synchronize_for_latency(device)
                elapsed_ms.append((time.perf_counter() - start) * 1000.0)
                forward_calls_per_iteration.append(forward_calls)
    finally:
        if was_training:
            model.train()

    values = np.asarray(elapsed_ms, dtype=np.float64)
    forward_calls = np.asarray(forward_calls_per_iteration, dtype=np.float64)
    total_sec = float(values.sum() / 1000.0)
    return {
        "latency_scope": "full_model_forward_calls_only",
        "architecture": ARCHITECTURE,
        "batch_size": batch_size,
        "seq_len": seq_len,
        "warmup_steps": warmup_steps,
        "measure_steps": measure_steps,
        "halt_max_steps": halt_max_steps,
        "initial_carry_timed": False,
        "forward_calls_mean": float(forward_calls.mean()),
        "forward_calls_min": int(forward_calls.min()),
        "forward_calls_max": int(forward_calls.max()),
        "precision": EFFECTIVE_PRECISION,
        "device": str(device),
        "cuda_synchronized": bool(device.type == "cuda" and torch.cuda.is_available()),
        "latency_ms_mean": float(values.mean()),
        "latency_ms_std": float(values.std()),
        "latency_ms_min": float(values.min()),
        "latency_ms_p50": float(np.percentile(values, 50)),
        "latency_ms_p95": float(np.percentile(values, 95)),
        "latency_ms_max": float(values.max()),
        "throughput_examples_per_sec": float(batch_size * measure_steps / total_sec) if total_sec > 0 else None,
    }


In [ ]:
def make_bert_source_config() -> Tuple[str, str]:
    return BERT_MODEL_NAME, str(BERT_LOCAL_PATH.expanduser())

def base_trm_config(pos_encodings: str) -> Dict[str, object]:
    return {
        "batch_size": LATENCY_BATCH_SIZE,
        "seq_len": SEQ_LEN,
        "vocab_size": VOCAB_SIZE,
        "H_cycles": H_CYCLES,
        "L_cycles": L_CYCLES,
        "H_layers": L_LAYERS,
        "L_layers": L_LAYERS,
        "hidden_size": HIDDEN_SIZE,
        "expansion": EXPANSION,
        "num_heads": NUM_HEADS,
        "pos_encodings": pos_encodings,
        "halt_max_steps": HALT_MAX_STEPS,
        "halt_exploration_prob": HALT_EXPLORATION_PROB,
        "forward_dtype": MODEL_FORWARD_DTYPE,
        "no_ACT_continue": True,
        "num_segment_types": 3,
    }

def build_model_and_config(architecture: str) -> Tuple[nn.Module, Dict[str, object]]:
    encoder_name, encoder_local_path = make_bert_source_config()

    if architecture == "bert_encoder_trm":
        model_config = base_trm_config(pos_encodings="none")
        model_config.update({
            "architecture": architecture,
            "encoder_name": encoder_name,
            "encoder_local_path": encoder_local_path,
            "freeze_encoder": FREEZE_ENCODER,
            "add_trm_segment_embeddings": ADD_TRM_SEGMENT_EMBEDDINGS,
        })
        model = TrmReranker({key: value for key, value in model_config.items() if key != "architecture"})
        return model, model_config

    if architecture == "bert_embeddings_trm":
        pretrained_embedding_model_name = encoder_local_path or BERT_MODEL_NAME
        model_config = base_trm_config(pos_encodings="rope")
        model_config.update({
            "architecture": architecture,
            "input_embedding_backend": "bert_word_embeddings_projected",
            "pretrained_embedding_model_name": pretrained_embedding_model_name,
            "freeze_pretrained_word_embeddings": FREEZE_PRETRAINED_WORD_EMBEDDINGS,
            "pretrained_word_embedding_projection": PRETRAINED_WORD_EMBEDDING_PROJECTION,
        })
        model = TrmReranker({key: value for key, value in model_config.items() if key != "architecture"})
        return model, model_config

    if architecture == "bert_encoder_rerank_head":
        model_config = {
            "architecture": architecture,
            "seq_len": SEQ_LEN,
            "vocab_size": VOCAB_SIZE,
            "encoder_name": encoder_name,
            "encoder_local_path": encoder_local_path,
            "freeze_encoder": FREEZE_ENCODER,
            "dropout": SCORING_HEAD_DROPOUT,
        }
        model = BertEncoderOnlyReranker(
            encoder_name=encoder_name,
            encoder_local_path=encoder_local_path,
            freeze_encoder=FREEZE_ENCODER,
            dropout=SCORING_HEAD_DROPOUT,
        )
        return model, model_config

    if architecture == "vanilla_transformer":
        model_config = {
            "architecture": architecture,
            "batch_size": LATENCY_BATCH_SIZE,
            "seq_len": SEQ_LEN,
            "vocab_size": VOCAB_SIZE,
            "num_layers": VANILLA_TRANSFORMER_LAYERS,
            "hidden_size": HIDDEN_SIZE,
            "expansion": EXPANSION,
            "num_heads": NUM_HEADS,
            "pos_encodings": "rope",
            "halt_max_steps": HALT_MAX_STEPS,
            "forward_dtype": MODEL_FORWARD_DTYPE,
            "num_segment_types": 3,
        }
        model = VanillaTransformerReranker({key: value for key, value in model_config.items() if key != "architecture"})
        return model, model_config

    raise ValueError(f"Unknown architecture={architecture!r}")

model, model_config = build_model_and_config(ARCHITECTURE)
model = model.to(device)
batch = make_synthetic_batch(LATENCY_BATCH_SIZE, SEQ_LEN, VOCAB_SIZE, device)

latency_result = measure_forward_latency(
    model=model,
    batch=batch,
    warmup_steps=LATENCY_WARMUP_STEPS,
    measure_steps=LATENCY_MEASURE_STEPS,
)

summary = {
    "architecture": ARCHITECTURE,
    "model_config": model_config,
    "total_parameters": count_model_parameters(model),
    "trainable_parameters": count_trainable_parameters(model),
    "device": str(device),
    "precision": EFFECTIVE_PRECISION,
    "latency_result": latency_result,
}

print(json.dumps(latency_result, indent=2))
print(json.dumps({
    "architecture": ARCHITECTURE,
    "total_parameters": summary["total_parameters"],
    "trainable_parameters": summary["trainable_parameters"],
}, indent=2))

output_path = OUTPUT_DIR / f"{ARCHITECTURE}_forward_latency_batch1.json"
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(f"Saved latency summary: {output_path}")

summary
